In [48]:
import pandas as pd

In [49]:
order = pd.read_excel("excel_file/List_of_Orders.xlsx")
order.head()

,Order ID,Order Date,CustomerName,State,City
0,B-25601,2018-01-04 00:00:00,Bharat,Gujarat,Ahmedabad
1,B-25602,2018-01-04 00:00:00,Pearl,Maharashtra,Pune
2,B-25603,2018-03-04 00:00:00,Jahan,Madhya Pradesh,Bhopal
3,B-25604,2018-03-04 00:00:00,Divsha,Rajasthan,Jaipur
4,B-25605,2018-05-04 00:00:00,Kasheen,West Bengal,Kolkata


In [50]:
order_details = pd.read_excel("excel_file/Order_Details.xlsx")
order_details.head()

,Order ID,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,1275,-1148,7,Furniture,Bookcases
1,B-25601,66,-12,5,Clothing,Stole
2,B-25601,8,-2,3,Clothing,Hankerchief
3,B-25601,80,-56,4,Electronics,Electronic Games
4,B-25602,168,-111,2,Electronics,Phones


In [51]:
sales_target = pd.read_excel("excel_file/Sales_target.xlsx")
sales_target.head()

,Month of Order Date,Category,Target
0,2026-04-18,Furniture,10400
1,2026-05-18,Furniture,10500
2,2026-06-18,Furniture,10600
3,2026-07-18,Furniture,10800
4,2026-08-18,Furniture,10900


### Part 1: Sales and Profitability Analysis

1. Merge the List of Orders and Order Details datasets on the basis of Order ID. Calculate the total sales (Amount) for each category across all orders.

In [52]:
merge_1 = order.merge(order_details, on = 'Order ID')
merge_1.head()

,Order ID,Order Date,CustomerName,State,City,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,2018-01-04 00:00:00,Bharat,Gujarat,Ahmedabad,1275,-1148,7,Furniture,Bookcases
1,B-25601,2018-01-04 00:00:00,Bharat,Gujarat,Ahmedabad,66,-12,5,Clothing,Stole
2,B-25601,2018-01-04 00:00:00,Bharat,Gujarat,Ahmedabad,8,-2,3,Clothing,Hankerchief
3,B-25601,2018-01-04 00:00:00,Bharat,Gujarat,Ahmedabad,80,-56,4,Electronics,Electronic Games
4,B-25602,2018-01-04 00:00:00,Pearl,Maharashtra,Pune,168,-111,2,Electronics,Phones


In [53]:
df_1 = merge_1.groupby('Category').agg(
    {
        'Amount':'sum'
    }
)
df_1

,Amount
Category,
Clothing,139054
Electronics,165267
Furniture,127181


2. For each category, calculate the average profit per order and total profit margin (profit as a percentage of Amount).

In [54]:
df_2 = merge_1.groupby('Category').agg(
    {
        'Profit':'sum',
        'Amount':'sum',
        'Quantity':'sum',
        'Order ID' : 'nunique',
        # 'Order ID' : 'count'
    }
)

df_2['Profit_Margin_%'] = (df_2['Profit'] / df_2['Amount']) * 100
df_2['Avg Profit per Order'] = (df_2['Profit'] / df_2['Order ID'])
df_2[['Avg Profit per Order', 'Profit_Margin_%']].round(2)

,Avg Profit per Order,Profit_Margin_%
Category,,
Clothing,28.40,8.03
Electronics,51.44,6.35
Furniture,12.35,1.81


3. Identify the top-performing and underperforming categories based on these metrics.

In [55]:
df_3 = df_2[['Avg Profit per Order', 'Profit_Margin_%', 'Profit', 'Amount', 'Order ID']].round(2)
df_3.columns = ['Avg Profit', 'Profit Margin %', 'Total Profit', 'Total Sales', 'Total Orders']
df_3

,Avg Profit,Profit Margin %,Total Profit,Total Sales,Total Orders
Category,,,,,
Clothing,28.40,8.03,11163,139054,393
Electronics,51.44,6.35,10494,165267,204
Furniture,12.35,1.81,2298,127181,186


In [56]:
# sub-category level
df_4 = merge_1.groupby(['Category','Sub-Category']).agg(
    {
        'Amount':'sum',
        'Profit':'sum'
    }
)
df_4['Margin_%'] = (df_4['Profit'] / df_4['Amount'] * 100).round(2)
df_4.sort_values(['Category','Profit'])

Amount  Profit  Margin_%
Category    Sub-Category                              
Clothing    Kurti               3361     181      5.39
            Skirt               1946     235     12.08
            Leggings            2106     260     12.35
            Saree              53511     352      0.66
            Shirt               7555    1131     14.97
            T-shirt             7382    1500     20.32
            Hankerchief        14608    2098     14.36
            Stole              18546    2559     13.80
            Trousers           30039    2847      9.48
Electronics Electronic Games   39168   -1236     -3.16
            Phones             46119    2207      4.79
            Accessories        21728    3559     16.38
            Printers           58252    5964     10.24
Furniture   Tables             22614   -4011    -17.74
            Chairs             34222     577      1.69
            Furnishings        13484     844      6.26
            Bookcases          56861    4888      8.60

### Part 2: Target Achievement Analysis

1. Using the Sales Target dataset, calculate the percentage change in target sales for the Furniture category month-over-month.

In [57]:
# month column is stored as Apr-18, May-18 ... in excel, got parsed wrongly (2026-04-18)
# actual month = month part, actual year = 2000 + day part
sales_target['Month'] = pd.to_datetime(
    dict(year = 2000 + sales_target['Month of Order Date'].dt.day,
         month = sales_target['Month of Order Date'].dt.month,
         day = 1)
).dt.strftime('%b-%Y')

furniture = sales_target[sales_target['Category'] == 'Furniture'].reset_index(drop=True)
furniture['MoM_%_Change'] = (furniture['Target'].pct_change() * 100).round(2)
furniture[['Month', 'Target', 'MoM_%_Change']]

,Month,Target,MoM_%_Change
0,Apr-2018,10400,NaN
1,May-2018,10500,0.96
2,Jun-2018,10600,0.95
3,Jul-2018,10800,1.89
4,Aug-2018,10900,0.93
5,Sep-2018,11000,0.92
6,Oct-2018,11100,0.91
7,Nov-2018,11300,1.80
8,Dec-2018,11400,0.88
9,Jan-2019,11500,0.88


2. Target vs actual furniture sales by month.

In [58]:
# order date also mixed up in excel (some rows month-first, some dd-mm-yyyy text)
def fix_date(d):
    if isinstance(d, str):
        return pd.to_datetime(d, dayfirst=True)
    return pd.Timestamp(year=d.year, month=d.day, day=d.month)

order['Order Date'] = order['Order Date'].apply(fix_date)
merge_1 = order.merge(order_details, on = 'Order ID')

actual = merge_1[merge_1['Category'] == 'Furniture'].groupby(
    merge_1['Order Date'].dt.strftime('%b-%Y'), sort=False
).agg({'Amount':'sum'})

df_5 = furniture[['Month','Target']].merge(actual, left_on='Month', right_index=True)
df_5['Achievement_%'] = (df_5['Amount'] / df_5['Target'] * 100).round(2)
df_5

,Month,Target,Amount,Achievement_%
0,Apr-2018,10400,8121,78.09
1,May-2018,10500,6220,59.24
2,Jun-2018,10600,5532,52.19
3,Jul-2018,10800,3483,32.25
4,Aug-2018,10900,9538,87.50
5,Sep-2018,11000,8704,79.13
6,Oct-2018,11100,6766,60.95
7,Nov-2018,11300,15165,134.20
8,Dec-2018,11400,9474,83.11
9,Jan-2019,11500,21257,184.84


### Part 3: Regional Performance Insights

1. From the List of Orders dataset, identify the top 5 states with the highest order count. For each of these states, calculate the total sales and average profit.

In [59]:
top_states = order['State'].value_counts().head(5)
top_states

State
Madhya Pradesh    101
Maharashtra        90
Rajasthan          32
Gujarat            27
Punjab             25
Name: count, dtype: int64

In [60]:
df_6 = merge_1[merge_1['State'].isin(top_states.index)].groupby('State').agg(
    {
        'Order ID':'nunique',
        'Amount':'sum',
        'Profit':'sum'
    }
)
df_6['Avg Profit'] = (df_6['Profit'] / df_6['Order ID']).round(2)
df_6['Profit_Margin_%'] = (df_6['Profit'] / df_6['Amount'] * 100).round(2)
df_6.sort_values('Order ID', ascending=False)

,Order ID,Amount,Profit,Avg Profit,Profit_Margin_%
State,,,,,
Madhya Pradesh,101,105140,5551,54.96,5.28
Maharashtra,90,95348,6176,68.62,6.48
Rajasthan,32,21149,1257,39.28,5.94
Gujarat,27,21058,465,17.22,2.21
Punjab,25,16786,-609,-24.36,-3.63


2. Regional disparities — city level.

In [61]:
df_7 = merge_1[merge_1['State'].isin(top_states.index)].groupby(['State','City']).agg(
    {
        'Order ID':'nunique',
        'Amount':'sum',
        'Profit':'sum'
    }
)
df_7['Margin_%'] = (df_7['Profit'] / df_7['Amount'] * 100).round(2)
df_7.sort_values('Profit')

,,Order ID,Amount,Profit,Margin_%
State,City,,,,
Punjab,Chandigarh,16,12279,-1153,-9.39
Gujarat,Ahmedabad,17,14230,-880,-6.18
Rajasthan,Jaipur,19,10076,-753,-7.47
Madhya Pradesh,Delhi,3,2488,521,20.94
Punjab,Amritsar,9,4507,544,12.07
Madhya Pradesh,Bhopal,22,23583,871,3.69
Gujarat,Surat,10,6828,1345,19.70
Maharashtra,Mumbai,68,61867,1637,2.65
Rajasthan,Udaipur,13,11073,2010,18.15
